# BÀI 3: KẾT HỢP VÀ GỘP BẢNG DỮ LIỆU TRONG PANDAS
## Mục tiêu học tập:
1. Làm chủ `pd.merge()` để thay thế hoàn toàn `VLOOKUP`/`XLOOKUP` và `SQL JOIN`.
2. Sử dụng `pd.concat()` để gộp nối báo cáo nhiều tháng/chi nhánh (tương đương `SQL UNION`).
3. Xử lý trường hợp lệch tên cột (`left_on`, `right_on`) và trùng tên cột (`suffixes`).

### Ô 1: Chuẩn bị 2 bảng dữ liệu mẫu (Quản lý tài sản & Danh mục phòng ban)

In [1]:
import pandas as pd

# Bảng 1: Danh sách tài sản kiểm kê tại chi nhánh
df_tai_san = pd.DataFrame({
    "Ma_TS": ["TS01", "TS02", "TS03", "TS04", "TS05"],
    "Ten_TS": ["Máy tính Dell", "Máy in HP", "Máy photocopy", "Máy scan", "Laptop Lenovo"],
    "Ma_Phong": ["PB01", "PB02", "PB01", "PB03", "PB99"],  # PB99 là phòng mới chưa cập nhật danh mục
    "Nguyen_gia": [15_000_000, 8_500_000, 45_000_000, 6_000_000, 22_000_000]
})

# Bảng 2: Danh mục mã phòng ban và người quản lý
df_phong_ban = pd.DataFrame({
    "Ma_Phong": ["PB01", "PB02", "PB03", "PB04"],  # PB04 chưa có tài sản nào
    "Ten_Phong": ["Kế toán", "Tín dụng", "Khách hàng DN", "Hành chính - Quản trị"],
    "Truong_phong": ["Nguyễn Văn A", "Trần Thị B", "Lê Văn C", "Phạm Minh D"]
})

print("--- BẢNG TÀI SẢN ---")
display(df_tai_san)
print("\n--- BẢNG PHÒNG BAN ---")
display(df_phong_ban)

--- BẢNG TÀI SẢN ---


,Ma_TS,Ten_TS,Ma_Phong,Nguyen_gia
0,TS01,Máy tính Dell,PB01,15000000
1,TS02,Máy in HP,PB02,8500000
2,TS03,Máy photocopy,PB01,45000000
3,TS04,Máy scan,PB03,6000000
4,TS05,Laptop Lenovo,PB99,22000000



--- BẢNG PHÒNG BAN ---


,Ma_Phong,Ten_Phong,Truong_phong
0,PB01,Kế toán,Nguyễn Văn A
1,PB02,Tín dụng,Trần Thị B
2,PB03,Khách hàng DN,Lê Văn C
3,PB04,Hành chính - Quản trị,Phạm Minh D


### Ô 2: Ghép bảng chuẩn VLOOKUP với `how='left'`
> **Nguyên tắc:** Giữ trọn vẹn 100% dòng của bảng bên trái (`df_tai_san`). Tìm thông tin phòng ban tương ứng ghép vào. Nếu không tìm thấy (như mã `PB99`), Pandas tự động điền `NaN`.

In [ ]:
df_merge_left = pd.merge(df_tai_san, df_phong_ban, on="Ma_Phong", how="left")
df_merge_left

### Ô 3: So sánh các chế độ ghép nối (`inner` vs `outer`)
- `inner`: Chỉ lấy giao điểm (bỏ qua `TS05` phòng `PB99` và bỏ qua phòng `PB04` không có tài sản).
- `outer`: Lấy trọn vẹn cả hai bảng, đầy đủ mọi đối tượng kiểm kê và phòng ban.

In [ ]:
# 1. Inner Join
df_inner = pd.merge(df_tai_san, df_phong_ban, on="Ma_Phong", how="inner")
print("Số dòng inner:", len(df_inner))
display(df_inner)

# 2. Outer Join (Full)
df_outer = pd.merge(df_tai_san, df_phong_ban, on="Ma_Phong", how="outer")
print("\nSố dòng outer:", len(df_outer))
display(df_outer)

### Ô 4: Xử lý khi 2 bảng lệch tên cột khóa (`left_on`, `right_on`)
Tình huống thực tế: File tài sản đặt tên cột là `Ma_PB`, trong khi file danh mục nhân sự lại ghi là `Dept_Code`.

In [ ]:
df_tai_san_lech = df_tai_san.rename(columns={"Ma_Phong": "Ma_PB"})
df_phong_ban_lech = df_phong_ban.rename(columns={"Ma_Phong": "Dept_Code"})

df_lech_cot = pd.merge(
    df_tai_san_lech,
    df_phong_ban_lech,
    left_on="Ma_PB",
    right_on="Dept_Code",
    how="left"
)

# Xóa bớt 1 trong 2 cột khóa trùng lặp nếu cần
df_lech_cot = df_lech_cot.drop(columns=["Dept_Code"])
df_lech_cot

### Ô 5: Nối chồng báo cáo định kỳ với `pd.concat` (Nối dọc `axis=0`)
Tình huống thực tế: Gộp danh sách kiểm kê tài sản đợt 1 và đợt 2 thành một danh sách duy nhất.

In [ ]:
# Bảng kiểm kê Đợt 1
df_dot1 = pd.DataFrame({
    "Ma_TS": ["TS01", "TS02"],
    "Ten_TS": ["Máy tính Dell", "Máy in HP"],
    "Trang_thai": ["Tốt", "Cần bảo trì"]
})

# Bảng kiểm kê Đợt 2
df_dot2 = pd.DataFrame({
    "Ma_TS": ["TS03", "TS04"],
    "Ten_TS": ["Máy photocopy", "Máy scan"],
    "Trang_thai": ["Tốt", "Hỏng chờ thanh lý"]
})

# Nối dọc bằng pd.concat (ignore_index=True để tạo lại số thứ tự từ 0 đến hết)
df_tong_hop = pd.concat([df_dot1, df_dot2], ignore_index=True)
df_tong_hop